### 02_split
Splits dataset (from manifest.csv) into two:
* `enroll`: used to create templates for enrollment of user in the model
* `probe`: used to determine accuracy of model from the identity match

Input: 
* `data_processed/<dataset>/manifests/manifest_basic.csv`

Output:
* `data_processed/<dataset>/manifests/manifest_enroll.csv`
* `data_processed/<dataset>/manifests/manifest_probe.csv`

In [16]:
# ----- Imports and config -----
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import math

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [17]:
# ----- Helper functions -----

# convert absolute path to relative path from project root
def rel_from_abs(abs_path):
    return Path(abs_path).resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()

In [18]:
# ----- Manifest building for enrollment and probe -----

def manifest_enroll_probe(MANIFEST_FILE, POOL_TRAIN_VAL=False, ENROLL_FRAC=0.8, SEED=42, CAP_ENROLL=None):
    """
    Create manifest_enroll.csv for 80/20 enroll/probe split 
     - Existing manifest_basic.csv contains all images with person_id labels
     - For each person_id, randomly select 80% of images for 'enroll'
     - The remaining 20% of images are used for 'probe'
     - Writes manifest_enroll.csv and manifest_probe.csv
     - This will be used for identity evaluation later
    """

    # Read manifest
    df = pd.read_csv(MANIFEST_FILE)
    print(f"Read manifest: {len(df)} rows from {rel_from_abs(MANIFEST_FILE)}")

    # Filter to train+val (or just train) before grouping
    if POOL_TRAIN_VAL:
        df = df[df['split'].isin(['train', 'val'])].reset_index(drop=True)
    else:
        df = df[df['split'] == 'train'].reset_index(drop=True)
    print(f"Using {len(df)} rows after filtering splits (pool_train_val={POOL_TRAIN_VAL})")

    # Prepare containers
    enroll_list = []
    probe_list = []

    # Group by person_id
    for person_id, g in df.groupby('person_id', sort=True):
        g = g.reset_index(drop=True)
        n = len(g)
        # deterministic RNG per person (mix global seed + person hash)
        # use hashlib to get consistent hash across runs
        seed = (SEED + int(hashlib.sha1(person_id.encode('utf8')).hexdigest()[:8], 16)) & 0xFFFFFFFF
        rng = np.random.RandomState(seed)
        perm = rng.permutation(n)
        # determine number of enroll images (ensure at least 1 if n>=1)
        if n == 1:
            n_enroll = 1
        else:
            n_enroll = max(1, int(math.floor(n * ENROLL_FRAC)))
        if CAP_ENROLL is not None:
            n_enroll = min(n_enroll, CAP_ENROLL)
        enroll_idx = perm[:n_enroll]
        probe_idx  = perm[n_enroll:]
        enroll_list.append(g.iloc[enroll_idx])
        if len(probe_idx) > 0:
            probe_list.append(g.iloc[probe_idx])

    # Concat results
    enroll_df = pd.concat(enroll_list, ignore_index=True) if enroll_list else pd.DataFrame(columns=df.columns)
    probe_df  = pd.concat(probe_list,  ignore_index=True) if probe_list  else pd.DataFrame(columns=df.columns)

    out_dir = Path("../data_processed/vggface2/manifests")
    out_dir.mkdir(parents=True, exist_ok=True)
    enroll_path = out_dir / "manifest_enroll.csv"
    probe_path  = out_dir / "manifest_probe.csv"

    enroll_df.to_csv(enroll_path, index=False)
    probe_df.to_csv(probe_path, index=False)

    print(f"Wrote enroll manifest: {enroll_path}  rows: {len(enroll_df)}  unique persons: {enroll_df['person_id'].nunique()}")
    print(f"Wrote probe  manifest: {probe_path}   rows: {len(probe_df)}  unique persons: {probe_df['person_id'].nunique()}")

    # Diagnostics
    single_image_persons = (df.groupby('person_id').size() == 1).sum()
    persons_no_probe = enroll_df['person_id'].nunique() - probe_df['person_id'].nunique()
    print(f"Persons with only one image in original data: {single_image_persons}")
    print(f"Persons present in enroll but not in probe: {persons_no_probe}")

    return enroll_df, probe_df

In [19]:
# ----- Run split to create enroll/probe manifests -----

# import and define paths
from scripts.config import DATA_PROCESSED

DS_DATASET = "vggface2"
MANIFEST_BASIC = DATA_PROCESSED / DS_DATASET / "manifests" / "manifest_basic.csv"

enroll_df, probe_df = manifest_enroll_probe(MANIFEST_BASIC, POOL_TRAIN_VAL=False, ENROLL_FRAC=0.8, SEED=42, CAP_ENROLL=None)

Read manifest: 197693 rows from data_processed/vggface2/manifests/manifest_basic.csv
Using 176398 rows after filtering splits (pool_train_val=False)
Wrote enroll manifest: ..\data_processed\vggface2\manifests\manifest_enroll.csv  rows: 140922  unique persons: 480
Wrote probe  manifest: ..\data_processed\vggface2\manifests\manifest_probe.csv   rows: 35476  unique persons: 480
Persons with only one image in original data: 0
Persons present in enroll but not in probe: 0


In [22]:
# Diagnostics: confirm train / val are disjoint and val is held out
from pathlib import Path

MAN_BASIC = Path(MANIFEST_BASIC)
df_basic = pd.read_csv(MAN_BASIC)
df_train = df_basic[df_basic["split"] == "train"]
df_val   = df_basic[df_basic["split"] == "val"]

print(f"Basic manifest rows: {len(df_basic)}")
print(f"Train rows: {len(df_train)}, unique persons: {df_train['person_id'].nunique()}")
print(f"Val rows:   {len(df_val)}, unique persons: {df_val['person_id'].nunique()}")
print("Overlap persons between train and val:", 
      len(set(df_train['person_id'].unique()).intersection(set(df_val['person_id'].unique()))))
print()
print(f"Enroll rows: {len(enroll_df)}, unique persons: {enroll_df['person_id'].nunique()}")
print(f"Probe rows:  {len(probe_df)}, unique persons: {probe_df['person_id'].nunique()}")
print("Any val rows present in enroll/probe (should be 0 or False):",
      (enroll_df['split'].isin(['val']).any(), probe_df['split'].isin(['val']).any()))

Basic manifest rows: 197693
Train rows: 176398, unique persons: 480
Val rows:   21295, unique persons: 60
Overlap persons between train and val: 0

Enroll rows: 140922, unique persons: 480
Probe rows:  35476, unique persons: 480
Any val rows present in enroll/probe (should be 0 or False): (np.False_, np.False_)


In [ ]:
# ----- Split manifest into train/val files -----

# Dataset's train and val splits are disjoint. 
# - These can be used for open-set identification
# - Split the manifests for easier use later
src = "../data_processed/vggface2/manifests/manifest_basic.csv"
df = pd.read_csv(src)
df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val   = df[df["split"] == "val"].reset_index(drop=True)
df_train.to_csv("../data_processed/vggface2/manifests/manifest_train.csv", index=False)
df_val.to_csv("../data_processed/vggface2/manifests/manifest_val.csv", index=False)
print(len(df_train), "train rows;", len(df_val), "val rows")

176398 train rows; 21295 val rows
